In [7]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import matplotlib as mpl
import plotconfig
from matplotlib.lines import Line2D

In [ ]:
# CHANGE THIS TO GENERATE FIGURE 12 OR FIGURE 21
figure = 12

if figure not in [12,21]:
    print("Invalid figure no.")
    sys.exit()

In [ ]:
rate = 100 if figure == 12 else 500

args = {
    "chaos": "real_kvm",
    "timestamp": ["2026", "2026052", "2026052", "2026052"],
    "cca": ["cubic", "bbr3", "bbr3-patched-new-new"],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel6-13-BBRv3", "zkernel6-13-BBRv3-patched_new_new"],

    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [rate],
    "delay_rtt": [10,20,30],
    "loadperc": [100, 90, 80, 70, 60, 50, 40, 30, 20, 10],
}

metric = "bits_per_second"

baselogpath = "../data"

In [9]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 2430
end 2430
bytes 2430
bits_per_second 2430
mbps_timeseries 2430
rttms_timeseries 2430
retransmits 2430
timestamp 2430
iteration 2430
cpu_host_total 2430
cpu_host_user 2430
cpu_host_system 2430
cpu_remote_total 2430
chaos 2430
os 2430
loadperc 2430
bdp 2430
setup 2430
cca 2430
cpus 2430
kernel 2430
mode 2430
loss 2430
rate 2430
delay_rtt 2430
buffer_size_bytes 2430
parallel 2430
socket_buffer 2430
app_buffer 2430
n 2430
sysctl_cmd 2430
vm 2430
bandwidth_delay_product 2430
loss_mode 2430
vms 2430
pacing 2430
hyperthreading 2430
tso 2430
qdisc 2430
hpet 2430
tsc 2430
hostq 2430
deadline_run 2430
deadline_period 2430
random_loss_rate 2430
gemodel_q 2430
original_cca 2430
test_cca 2430
default_qdisc 2430
json 2430


In [10]:
df["cca_generic"] = df["cca"].apply(lambda x: x.replace("-patched", "").replace("-new", ""))
df["patch_or_not"] = df["cca"].apply(lambda x: "apatched" if "-patched" in x else "original")
df["mbps"] = df["bits_per_second"]/1000000

replace_label = {
    "bbr": "BBRv1",
    "bbr2": "BBRv2",
    "bbr3": "BBRv3",
    "cubic": "Cubic"
}

In [11]:
def strip(df, savefig = False):    
    if savefig:
        mpl.use('agg')
    plotconfig.configure_conext()
    width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
    height = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)*(1/3.5)
    FIG_SIZE = (width, height)
    
    paletti = {"apatched": plotconfig.COLORS[1], "original": "gainsboro"}
    fig, axs = plt.subplots(1,1,figsize=FIG_SIZE, sharey=True,constrained_layout=True)

    df = df.sort_values(by=['patch_or_not'])
    cca = "bbr3"

    data_sorted=df[df["cca_generic"] == cca].sort_values(by=['patch_or_not','loadperc', 'delay_rtt'])
    extra_legend_patches = []
    
    for rtt_ in sorted(data_sorted["delay_rtt"].unique()):
        if rtt_ == 10:
            marker = "o"
        elif rtt_ == 20:
            marker = "v"
        elif rtt_ == 30:
            marker = "s"
        elif rtt_ == 40:
            marker = "P"
        extra_legend_patches.append(Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], markersize=3,mec=paletti["apatched"], marker=marker, label=f'{int(rtt_)}ms'))
        data_grouped = data_sorted[["loadperc", "mbps","patch_or_not", "delay_rtt"]].groupby(["loadperc", "patch_or_not", "delay_rtt"],as_index=False).median()
        sns.stripplot(ax=axs, data=data_grouped[data_grouped["delay_rtt"] == rtt_], x="loadperc", y="mbps", hue="patch_or_not", dodge=True, jitter=False, alpha=0.6, palette= paletti, zorder=0, legend=True if rtt_ == 10 else False, marker = marker, size=3,linewidth=0.1)        

    axs.tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-3)
    axs.vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)


    df_cubic = df[df["cca"] == "cubic"]
    df_cubic = df_cubic[df_cubic["kernel"] == "kernel6-1"]
    df_cubic = df_cubic.sort_values(by=['loadperc', "delay_rtt"])
    df_cubic = df_cubic[["loadperc", "mbps","patch_or_not", "delay_rtt"]].groupby(["loadperc", "patch_or_not", "delay_rtt"],as_index=False).median()
    for rtt_ in df_cubic["delay_rtt"].unique():
        if rtt_ == 10:
            marker = "o"
        elif rtt_ == 20:
            marker = "v"
        elif rtt_ == 30:
            marker = "s"
        elif rtt_ == 40:
            marker = "P"
        sns.stripplot(ax=axs, data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="loadperc", y="mbps", hue="patch_or_not", dodge=True, legend=False, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.7)


    han = [
        Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], mec="black", mew=0.1, marker="o", markersize=3, alpha=0.6, label=replace_label[cca]),
        Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], mec="black", mew=0.7, marker="o", markersize=1.8, alpha=0.9, label='Cubic'),
    ]
    leg1 = axs.legend(handles = han,loc="lower right", bbox_to_anchor=(1.1, 0.3), handlelength=1, framealpha=0.4,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
    axs.add_artist(leg1)

    h, l = axs.get_legend_handles_labels()
    for enu,hii in enumerate(h):
        hii.set_alpha(1)
        l[enu] = "original" if "original" in l[enu] else "patch"
    leg2 = axs.legend(handles=h, labels= l, loc="lower right", handlelength=0.9, bbox_to_anchor=(1.1,0.47), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)
    axs.add_artist(leg2)

    axs.legend(handles=extra_legend_patches, title="RTT", loc="lower right", handlelength=1.8, bbox_to_anchor=(1.1, 0), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)

    axs.set_xlabel("Hypervisor CPU load [\%]",fontsize=plotconfig.FONT_SIZE-3)
    axs.xaxis.set_inverted(True)
    axs.set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-3)
    axs.set_ylim(-float(args["rate"][0])*0.05,int(args["rate"][0]))
    
    if savefig:
        fig.savefig(f"figures/figure_{'12' if rate == 100 else '21'}.pdf", format="pdf")
    else:
        plt.show()

<>:67: SyntaxWarning: invalid escape sequence '\%'
<>:67: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_242311/159886360.py:67: SyntaxWarning: invalid escape sequence '\%'
  axs.set_xlabel("Hypervisor CPU load [\%]",fontsize=plotconfig.FONT_SIZE-3)


In [12]:
strip(df, True)